# AgroLeaf AI — Kaggle training (MobileNetV3-Large)

**Before you click Run All**

1. Attach the private Kaggle Dataset **`riceleafdiseasebd-v3-320`**. It must contain `RiceLeafDiseaseBD_v3_320/` and `splits.csv` at its root. Build it with `python ml/prepare_dataset.py` (see `ml/kaggle/KAGGLE_RUNBOOK.md`).
2. In the right sidebar: **Accelerator = GPU T4 x 1**, **Internet = On**.
3. Optional: attach the prepared external datasets from `ml/EXTERNAL_DATA.md` and add their specs to `EXTERNAL` in the config cell.
4. **Run All.** The final cell creates `/kaggle/working/run1_for_app.zip` without `best.pt`.

This notebook follows `ml/kaggle/KAGGLE_RUNBOOK.md` exactly: train → calibrate → evaluate → export ONNX → parity test. Do not report any metric until `metrics-v1.json` exists; numbers in this notebook are outputs, never estimates.


In [ ]:
# Cell 2 — clone the repository and install dependencies (skip torch/torchvision if present)
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/AgroLeaf-AI")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/masumcit44/AgroLeaf-AI.git", str(REPO)],
        check=True,
    )

requirements = (REPO / "ml" / "requirements.txt").read_text(encoding="utf-8").splitlines()
packages = [line.strip() for line in requirements if line.strip() and not line.strip().startswith("#")]
skip = set()
for name in ("torch", "torchvision"):
    if importlib.util.find_spec(name) is not None:
        skip.add(name)
packages = [p for p in packages if p.split("==")[0].split(">=")[0].split("<")[0].strip() not in skip]
print("Installing:", packages)
if packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Repository ready:", REPO)


In [ ]:
# Cell 3 — configuration (edit only this cell)
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/AgroLeaf-AI")

# The zip has this structure: RiceLeafDiseaseBD_v3_320/ + splits.csv
DATA_ROOT = Path("/kaggle/input/riceleafdiseasebd-v3-320/RiceLeafDiseaseBD_v3_320")
SPLITS_CSV = Path("/kaggle/input/riceleafdiseasebd-v3-320/splits.csv")

RUN_DIR = Path("/kaggle/working/run1")   # all artefacts land here
ARCH = "mnv3_large"                      # fallback: "mnv3_small"
EPOCHS = 20

# One spec per external set: "name=<root>:<map file>" (see ml/EXTERNAL_DATA.md).
EXTERNAL = []

assert REPO.exists(), f"Repository not found: {REPO}"
assert DATA_ROOT.exists(), f"Dataset folder not found: {DATA_ROOT}"
assert SPLITS_CSV.exists(), f"splits.csv not found: {SPLITS_CSV}"
RUN_DIR.mkdir(parents=True, exist_ok=True)


def run(script, *args):
    cmd = [sys.executable, str(REPO / script), *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(REPO), check=True)


print("DATA_ROOT :", DATA_ROOT)
print("SPLITS_CSV:", SPLITS_CSV)
print("RUN_DIR   :", RUN_DIR)
print("ARCH      :", ARCH, "| EPOCHS:", EPOCHS)
print("EXTERNAL  :", EXTERNAL or "none")


In [ ]:
# Cell 4 — GPU check (fail loudly if there is no GPU)
import subprocess
import torch

smi = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
print(smi.stdout or smi.stderr)
assert torch.cuda.is_available(), (
    "No GPU detected. Set Settings > Accelerator to 'GPU T4 x 1' and restart the session, then Run All again."
)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 5 — train MobileNetV3-Large
run(
    "ml/train.py",
    "--data-root", DATA_ROOT,
    "--splits", SPLITS_CSV,
    "--out", RUN_DIR,
    "--arch", ARCH,
    "--epochs", EPOCHS,
    "--batch", 32,
    "--device", "cuda",
    "--num-workers", 4,
)


In [ ]:
# Cell 6 — temperature calibration and validation-chosen threshold
run(
    "ml/calibrate.py",
    "--run", RUN_DIR,
    "--data-root", DATA_ROOT,
    "--splits", SPLITS_CSV,
    "--target-precision", 0.90,
)


In [ ]:
# Cell 7 — internal test + every external set in EXTERNAL
eval_args = ["--run", RUN_DIR, "--data-root", DATA_ROOT, "--splits", SPLITS_CSV]
for spec in EXTERNAL:
    eval_args += ["--external", spec]
run("ml/evaluate.py", *eval_args)


In [ ]:
# Cell 8 — export ONNX (opset 17, fixed 1x3x224x224) and enforce the size limit
run("ml/export_onnx.py", "--run", RUN_DIR)


In [ ]:
# Cell 9 — PyTorch vs ONNX numerical parity (< 1e-3)
run(
    "ml/parity_test.py",
    "--run", RUN_DIR,
    "--data-root", DATA_ROOT,
    "--splits", SPLITS_CSV,
    "--samples", 20,
)


In [ ]:
# Cell 10 — show metrics-v1.json and every PNG produced by the run
import json as _json

from IPython.display import Image, Markdown, display

metrics_path = RUN_DIR / "metrics-v1.json"
if metrics_path.exists():
    display(Markdown("### metrics-v1.json"))
    print(_json.dumps(_json.loads(metrics_path.read_text(encoding="utf-8")), indent=2))
else:
    print("WARNING: metrics-v1.json not found")

for png in sorted(RUN_DIR.glob("*.png")):
    display(Markdown(f"### {png.name}"))
    display(Image(filename=str(png)))


In [ ]:
# Cell 11 — zip the run for the app (best.pt is excluded) and print download steps
import zipfile

zip_path = Path("/kaggle/working/run1_for_app.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and path.name != "best.pt":
            zf.write(path, path.relative_to(RUN_DIR.parent))

print(f"Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MB)")
print()
print("Download it:")
print("  1. Open the right sidebar > Output > /kaggle/working.")
print("  2. Download run1_for_app.zip (do not download best.pt).")
print("  3. Unzip and copy the files into the app exactly as listed in")
print("     ml/kaggle/KAGGLE_RUNBOOK.md section 5:")
print("       model.onnx -> frontend/public/model/model.onnx")
print("       class_index.json, preprocess.json, calibration.json, metrics-v1.json -> frontend/src/model/")
print("       PNGs -> frontend/public/model/")
